# الفصل 11: الترميز الموضعي والتطبيع وبنية المحوّل الكاملة

> اكتشفنا في الفصل السابق أن Self-Attention قوي جدًا، لكن له عيبًا خفيًا وخطيرًا: هو لا مبالٍ تمامًا بالترتيب (permutation-invariant).

- **الجزء:** 3 — المحوّلات الأساسية
- **الفصل:** 11
- **المستوى:** متقدم
- **زمن القراءة:** 25 دقيقة
- **المتطلبات:** الفصل 10: الانتباه الذاتي متعدد الرؤوس
- **الوسوم:** Positional Encoding، LayerNorm، Encoder، Decoder

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch11-transformer-architecture.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch11-transformer-architecture.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch11-transformer-architecture.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> اكتشفنا في الفصل السابق أن Self-Attention قوي جدًا، لكن له عيبًا خفيًا وخطيرًا: هو لا مبالٍ تمامًا بالترتيب (permutation-invariant).

> **بيانات الفصل**
>
> - **الجزء:** 3 — المحوّلات الأساسية
> - **المستوى:** متقدم
> - **زمن القراءة:** 25 دقيقة
> - **المتطلبات:** الفصل 10: الانتباه الذاتي متعدد الرؤوس


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Positional Encoding
- فهم LayerNorm
- فهم Encoder


## 1. لماذا يهمّك هذا الموضوع؟

اكتشفنا في الفصل السابق أن Self-Attention قوي جدًا، لكن له عيبًا خفيًا وخطيرًا: هو **لا مبالٍ تمامًا بالترتيب (permutation-invariant)**. إذا بدّلت ترتيب الكلمات في الجملة، فإن مصفوفة الانتباه ستُنتج نفس المجموعة من القيم لكن بترتيب مختلف فقط — النموذج لا يملك أي طريقة جوهرية ليعرف أن "الكلب عضّ الرجل" مختلفة عن "الرجل عضّ الكلب" لو حُذفت معلومات الترتيب. هذا الفصل يحل هذه المشكلة عبر **الترميز الموضعي (Positional Encoding)**، ثم يُكمل بناء اللبنات المتبقية الضرورية لتكديس طبقات Transformer بعمق واستقرار: **Layer Normalization** و**الاتصالات المتبقية (Residual Connections)**، لنصل أخيرًا لتجميع كل هذه المكونات في معمارية **Transformer الكاملة** كما وردت في الورقة الأصلية.

---


## 2. الخريطة الذهنية


```text
المشكلة: Self-Attention عمياء عن الترتيب
"الكلب عض الرجل" و"الرجل عض الكلب" → نفس مجموعة أوزان الانتباه (مُعاد ترتيبها فقط)

الحل: Positional Encoding (PE)
│
├── أضف للتضمين الأصلي متجهًا يحمل "معلومات الموضع"
│   input = token_embedding + positional_encoding
│
├── Sinusoidal PE (الورقة الأصلية، ثابت غير متعلَّم):
│   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
│   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
│   → يسمح بالاستقراء لأطوال غير مرئية أثناء التدريب
│
└── Learned PE (BERT، GPT-2): متجه موضعي قابل للتعلم لكل موضع
    → أبسط لكن محدود بالطول الأقصى أثناء التدريب

المشكلة الثانية: تدريب شبكات عميقة جدًا (12، 24، 96 طبقة) بثبات
│
├── Layer Normalization: طبّع كل متجه عبر أبعاده الخاصة
│   (مقابل Batch Norm الذي يُطبّع عبر الدفعة — غير مناسب للتسلسلات)
│
├── Residual Connections: output = Sublayer(x) + x
│   → يسمح للتدرج بالتدفق مباشرة عبر "طريق مختصر" حتى الطبقات الأولى
│
└── Pre-LN مقابل Post-LN: أين نضع LayerNorm بالضبط؟
    Post-LN (الأصلي): LN(x + Sublayer(x))   → أداء أعلى لكن تدريب أقل استقرارًا
    Pre-LN (الحديث):  x + Sublayer(LN(x))   → تدريب أكثر استقرارًا للنماذج العميقة جدًا

البنية الكاملة لطبقة Encoder واحدة:
x → LN → Self-Attention → + residual → LN → FeedForward → + residual → output

البنية الكاملة لطبقة Decoder واحدة:
x → LN → Masked Self-Attention → +residual
  → LN → Cross-Attention (مع مخرجات Encoder) → +residual
  → LN → FeedForward → +residual → output
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل أنك تقرأ جملة مكتوبة بأحرف مبعثرة على طاولة بلا أي ترتيب فيزيائي محفوظ — فقط كومة كلمات. لفهم الجملة، تحتاج ليس فقط لمعرفة الكلمات، بل **أين تقع كل كلمة بالضبط**. الترميز الموضعي هو إضافة "رقم تسلسلي" ذكي لكل كلمة، لكن بدلاً من رقم بسيط (1، 2، 3...) الذي قد يُسبب مشاكل عددية مع التسلسلات الطويلة، نستخدم نمطًا من موجات الجيب وجيب التمام بترددات مختلفة — تمامًا مثل كيفية تمثيل الوقت بعقارب الساعة (الثانية، الدقيقة، الساعة) حيث كل "عقرب" يتحرك بسرعة مختلفة ليُعطي تمثيلًا فريدًا لكل لحظة.

أما Layer Normalization والاتصالات المتبقية، فتخيّلهما كـ"حواجز أمان" تمنع انهيار أو انفجار الإشارات العددية وهي تمر عبر عشرات أو مئات الطبقات المتراصة — دون هذه الحواجز، يصبح تدريب الشبكات العميقة جدًا شبه مستحيل.

### 🟡 طبقة الآلية: Positional Encoding بالتفصيل

**الصيغة الجيبية (Sinusoidal) من الورقة الأصلية**:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

حيث $pos$ هو موضع الكلمة في التسلسل (0، 1، 2...)، و$i$ هو رقم البُعد ضمن متجه الترميز (من 0 إلى $d_{model}/2 - 1$).

**لماذا هذا التصميم بالذات؟** ثلاثة أسباب جوهرية:
1. **قيم محدودة دائمًا بين -1 و1**: لا مشاكل عددية مهما طال التسلسل.
2. **كل بُعد يتذبذب بتردد مختلف**: الأبعاد الأولى تتذبذب بسرعة (تلتقط فروقات موضعية دقيقة/محلية)، بينما الأبعاد الأخيرة تتذبذب ببطء شديد (تلتقط موقعًا عامًا/بعيد المدى) — مشابه تمامًا لعقارب الساعة والدقائق والساعات.
3. **خاصية الإزاحة الخطية**: يمكن إثبات أن $PE_{pos+k}$ يمكن التعبير عنه كدالة خطية من $PE_{pos}$ لأي إزاحة ثابتة $k$، مما يُسهّل على النموذج تعلم "المسافات النسبية" بين المواضع — خاصية رياضية أنيقة جدًا نابعة من متطابقات جمع الزوايا المثلثية.

**Layer Normalization بالتفصيل**:

على عكس Batch Normalization الذي يُطبّع كل "ميزة" (feature) عبر عينات الدفعة (batch)، فإن Layer Normalization يُطبّع كل **عينة على حدة** عبر أبعادها الخاصة (عبر $d_{model}$):

$$\text{LN}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

حيث:
$$\mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d}(x_i - \mu)^2$$

و$\gamma, \beta$ معاملان قابلان للتعلم (scale وshift) لكل بُعد.

**لماذا LayerNorm وليس BatchNorm في Transformer؟** في معالجة اللغة، أطوال التسلسلات متغيرة، وحجم الدفعة قد يكون صغيرًا جدًا (حتى 1 في الاستدلال)، وBatchNorm يعتمد على إحصائيات دقيقة عبر الدفعة (خصوصًا أثناء الاستدلال حيث تُستخدم إحصائيات متحركة محفوظة من التدريب) — وهذا غير موثوق مع تسلسلات متغيرة الطول والحشو (padding). LayerNorm، بتطبيعه لكل عينة بمعزل عن البقية، لا يعاني من هذه المشاكل إطلاقًا، ويناسب تمامًا بنية البيانات التسلسلية.

**Pre-LN مقابل Post-LN**:

الورقة الأصلية (2017) استخدمت **Post-LN**:
$$x_{out} = \text{LN}(x + \text{Sublayer}(x))$$

البحث اللاحق (خصوصًا GPT-2 فصاعدًا) أظهر أن **Pre-LN** أكثر استقرارًا للتدريب العميق جدًا:
$$x_{out} = x + \text{Sublayer}(\text{LN}(x))$$

السبب: في Post-LN، يمر التدرج عبر LayerNorm قبل الوصول للمسار المتبقي (residual path)، مما قد يُضعف إشارة التدرج في الشبكات العميقة جدًا (30+ طبقة). في Pre-LN، المسار المتبقي "نظيف" تمامًا (بدون أي تطبيع عليه)، فيضمن تدفقًا مباشرًا وقويًا للتدرجات من الطبقة الأخيرة حتى الأولى — على حساب أداء نهائي أقل قليلًا أحيانًا مقارنة بـ Post-LN الذي تدرّب بنجاح.

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import math

# ============================================================
# 1. Positional Encoding الجيبي الكامل
# ============================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 10000^(2i/d_model) محسوبة بصيغة أسية لأسباب استقرار عددي
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # الأبعاد الزوجية
        pe[:, 1::2] = torch.cos(position * div_term)   # الأبعاد الفردية

        pe = pe.unsqueeze(0)   # (1, max_len, d_model) لتسهيل البث على الدفعة
        self.register_buffer('pe', pe)   # ليست معاملًا قابلًا للتعلم، لكنها تُحفظ مع النموذج

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ x: (batch, seq_len, d_model) """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

# ============================================================
# 2. Layer Normalization من الصفر (لفهم الآلية حرفيًا)
# ============================================================
class LayerNormFromScratch(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_normalized + self.beta

# ============================================================
# 3. Position-wise Feed-Forward Network
# ============================================================
class PositionWiseFeedForward(nn.Module):
    """
    شبكة تغذية أمامية تُطبَّق بشكل مستقل على كل موضع (position)
    بنفس الأوزان، عادة بتوسيع البعد 4 أضعاف ثم إعادته.
    """
    def __init__(self, d_model: int, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

# ============================================================
# 4. طبقة Encoder كاملة (Pre-LN، النمط الحديث الأكثر استقرارًا)
# ============================================================
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        from_scratch_mha = MultiHeadAttention  # من الفصل العاشر
        self.self_attn = from_scratch_mha(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNormFromScratch(d_model)
        self.norm2 = LayerNormFromScratch(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # --- Pre-LN Self-Attention + Residual ---
        normed_x = self.norm1(x)
        attn_output, attn_weights = self.self_attn(normed_x, normed_x, normed_x, mask)
        x = x + self.dropout1(attn_output)

        # --- Pre-LN FeedForward + Residual ---
        normed_x = self.norm2(x)
        ff_output = self.feed_forward(normed_x)
        x = x + self.dropout2(ff_output)

        return x, attn_weights

# ============================================================
# 5. طبقة Decoder كاملة (Masked Self-Attn + Cross-Attn + FFN)
# ============================================================
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNormFromScratch(d_model)
        self.norm2 = LayerNormFromScratch(d_model)
        self.norm3 = LayerNormFromScratch(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None):
        # Masked Self-Attention (سببي)
        normed_x = self.norm1(x)
        self_attn_out, _ = self.masked_self_attn(normed_x, normed_x, normed_x, self_mask)
        x = x + self.dropout1(self_attn_out)

        # Cross-Attention: Q من الـ Decoder، K,V من الـ Encoder
        normed_x = self.norm2(x)
        cross_attn_out, cross_attn_weights = self.cross_attn(
            normed_x, encoder_output, encoder_output, cross_mask
        )
        x = x + self.dropout2(cross_attn_out)

        # FeedForward
        normed_x = self.norm3(x)
        ff_out = self.feed_forward(normed_x)
        x = x + self.dropout3(ff_out)

        return x, cross_attn_weights

# ============================================================
# 6. معمارية Transformer الكاملة (Encoder-Decoder)
# ============================================================
class FullTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_heads=8, num_layers=6, d_ff=2048, max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = LayerNormFromScratch(d_model)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

    def encode(self, src, src_mask):
        x = self.src_embedding(src) * math.sqrt(self.d_model)  # تحجيم تقليدي من الورقة الأصلية
        x = self.pos_encoding(x)
        x = self.dropout(x)
        for layer in self.encoder_layers:
            x, _ = layer(x, src_mask)
        return self.final_norm(x)

    def decode(self, tgt, encoder_output, tgt_mask, cross_mask):
        x = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        for layer in self.decoder_layers:
            x, cross_attn = layer(x, encoder_output, tgt_mask, cross_mask)
        return self.final_norm(x), cross_attn

    def forward(self, src, tgt, src_mask, tgt_mask, cross_mask):
        encoder_output = self.encode(src, src_mask)
        decoder_output, cross_attn = self.decode(tgt, encoder_output, tgt_mask, cross_mask)
        logits = self.output_projection(decoder_output)
        return logits, cross_attn


---


## 4. مثال محلول خطوة بخطوة

لنحسب الترميز الموضعي يدويًا لأول موضعين (pos=0, pos=1) بأبعاد صغيرة جدًا $d_{model}=4$ (أي $i \in \{0, 1\}$):

**للموضع pos=0**:

$$PE(0, 0) = \sin(0/10000^{0/4}) = \sin(0) = 0$$
$$PE(0, 1) = \cos(0/10000^{0/4}) = \cos(0) = 1$$
$$PE(0, 2) = \sin(0/10000^{2/4}) = \sin(0) = 0$$
$$PE(0, 3) = \cos(0/10000^{2/4}) = \cos(0) = 1$$

$$PE_0 = [0, 1, 0, 1]$$

**للموضع pos=1**:

$$PE(1, 0) = \sin(1/10000^{0/4}) = \sin(1) \approx 0.8415$$
$$PE(1, 1) = \cos(1/10000^{0/4}) = \cos(1) \approx 0.5403$$
$$PE(1, 2) = \sin(1/10000^{2/4}) = \sin(1/100) = \sin(0.01) \approx 0.01$$
$$PE(1, 3) = \cos(1/10000^{2/4}) = \cos(0.01) \approx 0.99995$$

$$PE_1 = [0.8415, 0.5403, 0.01, 0.99995]$$

**ملاحظة مهمة**: لاحظ أن البعدين الأولين ($i=0$) تغيّرا بشكل كبير بين الموضعين (من 0 إلى 0.84)، بينما البعدان الأخيران ($i=1$) تغيّرا بشكل طفيف جدًا (من 0 إلى 0.01). هذا يُظهر عمليًا كيف تلتقط الأبعاد المختلفة ترددات مختلفة: الأبعاد الأولى حسّاسة للتغيرات الموضعية المحلية الدقيقة، بينما الأبعاد الأخيرة تتغير ببطء لتلتقط السياق الموضعي العام البعيد المدى.

---


## 5. نصائح وحيل

> 💡 **اختبر PE بصريًا دائمًا**: ارسم مصفوفة PE كاملة (موضع × بُعد) كخريطة حرارية لأول 100 موضع — يجب أن ترى نمطًا موجيًا جميلًا يتغير تردده عبر الأبعاد. إذا بدت عشوائية تمامًا، فهناك خطأ في صيغة `div_term`.

> 💡 **Learned Positional Embeddings بديل بسيط وفعّال**: في النماذج الحديثة (BERT, GPT-2) استُخدمت متجهات موضعية قابلة للتعلم بدلاً من الصيغة الجيبية الثابتة — أبسط للتنفيذ، لكن محدودة بطول أقصى ثابت تم التدريب عليه (لا يمكن الاستقراء لتسلسلات أطول دون إعادة تدريب أو تقنيات خاصة مثل RoPE التي سنراها في الجزء الرابع).

> 💡 **تذكّر `register_buffer` وليس `nn.Parameter` للـ PE الثابت**: لأن الترميز الجيبي ليس قابلاً للتعلم، استخدم `register_buffer` بدلاً من `nn.Parameter` حتى لا يحاول المُحسِّن تحديث هذه القيم.

> 💡 **Pre-LN هو الخيار الافتراضي الآمن اليوم**: إذا كنت تبني نموذجًا من الصفر بأكثر من 12 طبقة، ابدأ بـ Pre-LN — كل نماذج LLM الحديثة تقريبًا (GPT-3 فصاعدًا، LLaMA، إلخ) تستخدمه لثبات التدريب الفائق.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **نسيان تحجيم التضمينات بـ $\sqrt{d_{model}}$**: الورقة الأصلية تضرب تضمينات الكلمات في $\sqrt{d_{model}}$ قبل جمعها مع PE، لجعل مقياس (scale) التضمينات مماثلًا لمقياس الترميز الموضعي (الذي قيمه محصورة بين -1 و1) — إغفال هذا يُنتج أداءً أضعف خصوصًا في المراحل الأولى من التدريب.

> ⚠️ **الخلط بين `.var(unbiased=True)` و`.var(unbiased=False)` في LayerNorm**: التطبيق القياسي لـ LayerNorm يستخدم التباين المنحاز (biased, بالقسمة على n وليس n-1)؛ استخدام `unbiased=True` بالخطأ يُنتج نتائج مختلفة قليلًا عن `nn.LayerNorm` الرسمية في PyTorch، مما قد يُسبب تناقضات عند تحميل أوزان مُدرَّبة مسبقًا.

> ⚠️ **بناء Post-LN وPre-LN بشكل غير متسق**: إذا كنت تُحمّل أوزانًا مُدرَّبة مسبقًا من نموذج استخدم Post-LN وتُطبّقها على معمارية Pre-LN (أو العكس)، فستحصل على نتائج كارثية رغم تطابق أسماء الطبقات ظاهريًا — تأكد دائمًا من مطابقة ترتيب العمليات بالضبط.

> ⚠️ **تجاهل الـ Dropout على أوزان الانتباه**: التطبيق الكامل يتضمن Dropout في عدة مواضع (بعد softmax في الانتباه، بعد FFN، على التضمينات) — إغفال أي منها يُضعف التعميم خصوصًا في النماذج الكبيرة المعرّضة للحفظ الزائد (overfitting) على بيانات تدريب محدودة.

---


## 7. جدول مقارنة

### Sinusoidal مقابل Learned Positional Encoding

| الخاصية | Sinusoidal PE | Learned PE |
|---|---|---|
| **قابلية التعلم** | ثابت، غير قابل للتعلم | قابل للتعلم بالكامل |
| **الاستقراء لأطوال أطول** | ممكن نظريًا (بامتداد الصيغة) | غير ممكن تجاوز الطول الأقصى المُدرَّب عليه |
| **عدد المعاملات الإضافية** | صفر | $max\_len \times d_{model}$ |
| **الاستخدام** | الورقة الأصلية، بعض نماذج الترجمة | BERT، GPT-2، معظم LLMs المبكرة |
| **البديل الحديث** | — | RoPE وALiBi (الجزء الرابع) تجاوزا كليهما |

### Pre-LN مقابل Post-LN

| الخاصية | Post-LN (الأصلي) | Pre-LN (الحديث) |
|---|---|---|
| **الصيغة** | $\text{LN}(x+\text{Sublayer}(x))$ | $x+\text{Sublayer}(\text{LN}(x))$ |
| **استقرار التدريب العميق** | أضعف (يحتاج warmup حذر جدًا) | أقوى بكثير، يسمح بعشرات/مئات الطبقات |
| **الأداء النهائي (بعد تدريب ناجح)** | غالبًا أعلى قليلًا | قريب جدًا، أحيانًا أقل بفارق ضئيل |
| **الحاجة لـ Learning Rate Warmup** | حرجة جدًا | أقل حساسية نسبيًا |
| **الاستخدام في النماذج الحديثة** | نادر الآن | المعيار شبه الحصري (GPT-3+, LLaMA, إلخ) |

---


## 8. ملخص أفضل الممارسات

- ✅ استخدم Pre-LN كخيار افتراضي لأي معمارية عميقة جديدة تبنيها من الصفر.
- ✅ تحقّق بصريًا من صحة الترميز الموضعي قبل دمجه في أي نموذج أكبر.
- ✅ اضبط $\epsilon$ في LayerNorm على قيمة صغيرة قياسية (عادة $10^{-5}$ أو $10^{-6}$) لتفادي القسمة على صفر دون التأثير على الدقة.
- ✅ طبّق Dropout بانتظام على: التضمينات، أوزان الانتباه بعد softmax، ومخرجات كل Sublayer قبل الجمع مع المسار المتبقي.
- ✅ استخدم `final_norm` (طبقة LayerNorm أخيرة) بعد آخر طبقة Encoder/Decoder عند استخدام Pre-LN — ضرورية لتطبيع المخرج النهائي الذي لم يمر بأي LN بعد آخر إضافة متبقية.

---


## 9. أسئلة مقابلات

**س1: لماذا تحتاج معمارية Transformer إلى ترميز موضعي بينما لا تحتاجه RNN؟**

**الإجابة النموذجية**: RNN تعالج التسلسل خطوة بخطوة بترتيب صريح (الخطوة t تعتمد حسابيًا على مخرج الخطوة t-1)، لذا فإن معلومات الترتيب **متأصلة** في آلية المعالجة نفسها. أما Self-Attention، فهي عملية متناظرة رياضيًا بالكامل (permutation-equivariant): لو بدّلت ترتيب المدخلات، فإن النتائج تُعاد ترتيبها فقط دون أي تغيير جوهري في القيم، لأن كل عملية (حساب Q,K,V، ضرب المصفوفات، softmax) تعامل كل موضع بشكل مستقل عن "رقمه الترتيبي" الفعلي. لذلك، وبدون أي إضافة خارجية، ستكون معمارية Transformer عمياء تمامًا عن ترتيب الكلمات — وهذا يستدعي حقن معلومات الموضع بشكل صريح عبر الترميز الموضعي.

**س2: اشرح لماذا تفضّل Transformer استخدام Layer Normalization بدلاً من Batch Normalization.**

**الإجابة النموذجية**: Batch Normalization يُطبّع كل "ميزة" (قناة أو بُعد) عبر كل عينات الدفعة، وهذا يفترض ضمنيًا أن إحصائيات الدفعة (المتوسط والتباين) مستقرة وممثّلة جيدًا — هذا يصبح مُشكِلًا في معالجة اللغة لسببين: (1) أطوال التسلسلات متغيرة وتحتاج حشوًا (padding) يُشوّه إحصائيات BatchNorm إذا لم يُعالَج بعناية فائقة، و(2) حجم الدفعة في بعض سيناريوهات الاستدلال قد يكون 1 فقط، مما يجعل إحصائيات BatchNorm غير موثوقة (رغم استخدام إحصائيات متحركة محفوظة، فهي قد لا تناسب كل مثال فرديًا). Layer Normalization، في المقابل، يُطبّع كل عينة (كل موضع في التسلسل) بمعزل تام عن بقية الدفعة، معتمدًا فقط على القيم عبر أبعاد $d_{model}$ لتلك العينة بالذات — هذا يجعله مستقلًا تمامًا عن حجم الدفعة وطول التسلسل، ومناسبًا تمامًا لبنية البيانات التسلسلية في معالجة اللغة.

**س3: ما الفرق بين Pre-LN وPost-LN، ولماذا تتجه النماذج الحديثة لاستخدام Pre-LN؟**

**الإجابة النموذجية**: في Post-LN (التصميم الأصلي لورقة 2017)، يُطبَّق LayerNorm **بعد** إضافة الاتصال المتبقي: $\text{LN}(x+\text{Sublayer}(x))$ — هذا يعني أن المسار المتبقي (residual path) ذاته يمر عبر تطبيع في كل طبقة، مما قد يُضعف إشارة التدرج أثناء الانتشار العكسي عبر عشرات الطبقات (كل LN يُدخل بعض "التشويش" على التدرج الخام). في Pre-LN، يُطبَّق LayerNorm **قبل** الدخول للـ Sublayer فقط: $x+\text{Sublayer}(\text{LN}(x))$ — هنا يبقى المسار المتبقي نقيًا تمامًا بلا أي تطبيع عليه مباشرة، فيضمن تدفقًا مباشرًا وقويًا جدًا للتدرجات من آخر طبقة حتى أول طبقة (مشابه لفكرة ResNet الأصلية). هذا الاستقرار الإضافي يسمح بتدريب نماذج بعشرات أو مئات الطبقات (مثل GPT-3 بـ96 طبقة) دون الحاجة لجدولة warmup دقيقة جدًا كما يتطلبه Post-LN، وهو السبب الرئيسي وراء تبنّي كل النماذج الكبيرة الحديثة تقريبًا لـ Pre-LN رغم تنازل بسيط أحيانًا في الأداء النهائي مقارنة بـ Post-LN الذي نجح تدريبه.

**س4: لماذا تُستخدم الدوال الجيبية (sin/cos) تحديدًا في الترميز الموضعي الأصلي، وليس مثلًا ترميزًا ثنائيًا (binary) بسيطًا للموضع؟**

**الإجابة النموذجية**: الترميز الثنائي البسيط (مثل تمثيل الرقم 5 كـ 101) غير مستمر (discontinuous) وغير سلس رياضيًا — تغييرات صغيرة في الموضع قد تُنتج قفزات كبيرة في بعض البتات، وهذا يصعب على شبكة عصبية قائمة على عمليات مستمرة (ضرب، جمع) أن تتعلم منه أنماطًا سلسة. الدوال الجيبية، بالمقابل، مستمرة وسلسة تمامًا، ومحدودة القيمة (بين -1 و1) بغض النظر عن حجم الموضع (لا مشاكل عددية مع التسلسلات الطويلة). والأهم: بفضل متطابقات جمع الزوايا المثلثية، يمكن إثبات أن $PE_{pos+k}$ هو دالة **خطية** من $PE_{pos}$ لأي إزاحة ثابتة k، مما يمنح النموذج قدرة أسهل على تعلم العلاقات **النسبية** بين المواضع (وليس فقط المواضع المطلقة) عبر تركيبات خطية بسيطة من أوزان الانتباه — خاصية مهمة جدًا لفهم بنية اللغة التي تعتمد على المسافات النسبية بين الكلمات أكثر من مواضعها المطلقة.

---


## 10. تمرين عملي

1. نفّذ `SinusoidalPositionalEncoding` وارسم خريطة حرارية لأول 50 موضع × 64 بُعد — لاحظ التردد المتناقص عبر الأبعاد.
2. أثبت عدديًا (وليس تحليليًا فقط) خاصية الإزاحة الخطية: احسب $PE_5$ ثم $PE_8$، وتحقق أن $PE_8$ يمكن التعبير عنه تقريبًا كتركيبة خطية بسيطة من $PE_5$ ومعاملات ثابتة مرتبطة بالإزاحة k=3.
3. ابنِ `TransformerEncoderLayer` كاملة وطبّقها على تسلسل عشوائي بطول 10، وتحقق من تطابق شكل المدخل والمخرج (`(batch, seq_len, d_model)`).
4. قارن تجريبيًا استقرار التدريب (بمراقبة قيمة الخسارة في أول 100 خطوة) بين نسخة Pre-LN ونسخة Post-LN من نفس `TransformerEncoderLayer` على مهمة تصنيف بسيطة بمعدل تعلم مرتفع نسبيًا — لاحظ أيهما أكثر مقاومة للتباعد (divergence).

---


## 11. مراجع للتعمّق

- Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS. (القسم 3.5 لـ Positional Encoding)
- Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). *Layer Normalization*. arXiv.
- Xiong, R., et al. (2020). *On Layer Normalization in the Transformer Architecture*. ICML. (التحليل الرياضي الكامل لـ Pre-LN مقابل Post-LN)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المحوّلات الأساسية / الفصل 11: الترميز الموضعي والتطبيع وبنية المحوّل الكاملة.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
